In [8]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

In [2]:
MODEL=os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

load_dotenv()

llm=ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model=MODEL
)

def ask(system_prompt: str, user_prompt: str)->str:
    return llm.invoke(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ]
    ).content


## Langgraph States

In [19]:
class TicketState(TypedDict):
    text: str
    normalized: str
    category: str

## Langgraph Nodes (Defined as functions)

In [27]:
def normalize(state: TicketState)->dict:
    return {
        "normalized": " ".join(state["text"].lower().split())
    }

In [28]:
def classify(state: TicketState)->dict:
    category=ask(
        system_prompt="Classify as billing, technical, support or other. Output one lowercase word.",
        user_prompt=state["normalized"].strip().lower()
    )

    return {
        "category": category if category in ["billing", "technical", "support", "other"] else "other"
    }

## Creating the Graph

In [29]:
builder=StateGraph(state_schema=TicketState)

builder.add_node("normalize", normalize)
builder.add_node("classify", classify)

builder.add_edge(start_key=START, end_key="normalize")
builder.add_edge(start_key="normalize", end_key="classify")
builder.add_edge(start_key="classify", end_key=END)

## Compiling & Running the Graph

In [30]:
graph=builder.compile()

graph.invoke({
    "text": "I need to cancel my subscription."
})

{'text': 'I need to cancel my subscription.',
 'normalized': 'i need to cancel my subscription.',
 'category': 'billing'}